In [ ]:
import sys
sys.path.append("../preprocess")
from qbiolip import load_db
import torch
import pandas
from plotly import express as xp
from plotly import graph_objects as go

df = load_db("../../databases/QBioLiP/qbiolip.csv", "../../databases/ec_uniprot_ids.txt")
df = df.loc[~df.group.isna()]
df["equibind_rmsd"] = None
df["equibind_trimed_rmsd"] = None

outputs = torch.load("/home/daniil/Desktop/Ph.D/EquiBind/runs/flexible_self_docking/predictions_RDKitFalse.pt")
targets = outputs["targets"]
predictions = outputs["initial_predictions"]
names = outputs["names"]

for i,name in enumerate(names):
    rmsd = (targets[i]-predictions[i]).pow(2).sum(dim=1).mean().sqrt()
    if "trimed" not in name:
        df.loc[df.ligand_detail == name, ["equibind_rmsd"]] = rmsd
    else:
        df.loc[df.ligand_detail == name.replace("_trimed", ""), ["equibind_trimed_rmsd"]] = rmsd

equib_df = df.loc[(~df.equibind_trimed_rmsd.isna()) & (~df.equibind_rmsd.isna())]
equib_df.sort_values(by="equibind_rmsd", inplace=True)
equib_df.reset_index(drop=True,inplace=True)
colors = pandas.Series(["red"]*equib_df.shape[0])
colors[equib_df.loc[equib_df.group=="B"].index] = "blue"
colors[equib_df.loc[equib_df.group=="C"].index] = "green"

# plot overall rmsd:
fig = xp.bar(
    equib_df, x="ligand_detail", y="equibind_rmsd",
    title="EquiBind RMSD on 90 enzyme-ligand complexes wit implicit hydrogenation",
    color=colors,
    color_discrete_map="identity",
    category_orders={"ligand_detail": equib_df.ligand_detail.tolist()},
    log_y=True,
)
fig.update_layout(
    xaxis_title="Name of the complex",
    yaxis_title="RMSD, Logarithmic scale",
)
fig.show()

# plot rmsd per group
x = equib_df.group.unique()
y = [equib_df.loc[equib_df.group==group].equibind_rmsd.mean() for group in x]
fig = go.Figure([
    go.Bar(x=x, y=y)
])
fig.update_layout(
    title="Mean RMSD for each categories"
)
fig.show()
# plot trimed rmsd:
equib_df.sort_values(by="equibind_trimed_rmsd", inplace=True)
equib_df.reset_index(drop=True,inplace=True)
fig = xp.bar(
    equib_df, x="ligand_detail", y="equibind_trimed_rmsd",
    title="EquiBind trimed RMSD on 90 enzyme-ligand complexes wit implicit hydrogenation",
    color=colors,
    color_discrete_map="identity",
    category_orders={"ligand_detail": equib_df.ligand_detail.tolist()},
    log_y=True,
)
fig.update_layout(
    xaxis_title="Name of the complex",
    yaxis_title="RMSD, Logarithmic scale",
)
fig.show()

# plot rmsd per group
x = equib_df.group.unique()
y = [equib_df.loc[equib_df.group==group].equibind_trimed_rmsd.mean() for group in x]
fig = go.Figure([
    go.Bar(x=x, y=y)
])
fig.update_layout(
    title="Mean RMSD for each categories"
)
fig.show()

def count_res(x):
    return len(x.split("-"))
equib_df["num_atoms"] = equib_df.lig_atoms.str.len()

fig = xp.histogram(equib_df, x="num_atoms",
    title="Number of atoms in ligands"
)
fig.show()

fig = xp.histogram(equib_df, x="rec_len_resi",
    title="Number of residues in assemblies", log_y=True
)
fig.show()
# plot rmsd per group
x = equib_df.group.unique()
y = [equib_df.loc[equib_df.group==group].equibind_trimed_rmsd.mean() for group in x]
fig = go.Figure([
    go.Bar(x=x, y=y)
])
fig.update_layout(
    title="Mean RMSD for each categories"
)
fig.show()

def count_res(x):
    return len(x.split("-"))
equib_df["num_atoms"] = equib_df.lig_atoms.str.len()

fig = xp.histogram(equib_df, x="num_atoms",
    title="Number of atoms in ligands"
)
fig.show()

fig = xp.histogram(equib_df, x="rec_len_resi",
    title="Number of residues in assemblies", log_y=True
)
fig.show()